
# NeoLM — Small Language Model Comparison

Compare [NeoLM-Small-Base](https://huggingface.co/MaagDeveloper/NeoLM-Small-Base) with similarly sized pretrained decoder-only language models.

## Models

- NeoLM-Small-Base — 61.4M
- GPT2-Mini — 39M
- Pythia-70M — 70.4M
- DistilGPT2 — 82M

## Evaluation

1. **Text Generation:** Reuse NeoLM's saved prompts and sampling configurations.
2. **HellaSwag:** Evaluate 200 identical validation examples.
3. **Inference Performance:** Measure generation throughput.

## Methodology

- Base models only; no instruction tuning.
- Same input texts and sampling configurations.
- Model-specific tokenizers.
- No fine-tuning.
- Run on the same GPU.

## Limitations

Training data, token counts, and training methods differ across models.

HellaSwag uses a small sample and custom scoring implementation. Results are exploratory, not official leaderboard scores.

Generation quality is evaluated qualitatively.


In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from transformers import set_seed
import pandas as pd
import re
from datasets import load_dataset
import time
import numpy as np
import torch.nn.functional as F
import torch
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
RESULTS_DIR = Path("results")

MODEL_IDS = {
    "GPT2-Mini": "erwanf/gpt2-mini",
    "Pythia-70M": "EleutherAI/pythia-70m",
    "DistilGPT2": "distilbert/distilgpt2",
}

HELLASWAG_SAMPLE_SIZE = 200
SEED = 1337

# Reuse the original NeoLM benchmark
with open(
    RESULTS_DIR / "neolm_generation.json",
    encoding="utf-8",
) as f:
    baseline = json.load(f)

generation_cases = baseline["results"]

unique_prompts = list(dict.fromkeys(
    (r["category"], r["prompt"])
    for r in generation_cases
))

configs = list(dict.fromkeys(
    r["config_name"]
    for r in generation_cases
))

print("=" * 60)
print("SMALL LANGUAGE MODEL COMPARISON")
print("=" * 60)
print(f"Models:               {len(MODEL_IDS) + 1}")
print(f"Generation prompts:   {len(unique_prompts)}")
print(f"Sampling configs:     {len(configs)}")
print(f"Generations/model:    {len(generation_cases)}")
print(f"HellaSwag examples:   {HELLASWAG_SAMPLE_SIZE}")
print("=" * 60)

print("\nComparison models:")
for name, model_id in MODEL_IDS.items():
    print(f"  {name:15} {model_id}")

In [ ]:
def run_generation_benchmark(model, tokenizer, model_name, model_id):
    results = []

    context_limit = model.config.max_position_embeddings

    print(f"\n{'=' * 60}")
    print(f"GENERATION BENCHMARK — {model_name}")
    print(f"{'=' * 60}")
    print(f"Model: {model_id}")
    print(f"Cases: {len(generation_cases)}")
    print(f"{'=' * 60}")

    for case in tqdm(generation_cases, desc=model_name):
        inputs = tokenizer(
            case["prompt"],
            return_tensors="pt",
        ).to(model.device)

        prompt_tokens = inputs["input_ids"].shape[1]

        max_new_tokens = min(
            baseline["max_new_tokens"],
            context_limit - prompt_tokens,
        )

        assert max_new_tokens > 0

        set_seed(case["seed"])

        if model.device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                **case["generation_config"],
            )

        if model.device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        generated_ids = outputs[0, prompt_tokens:]
        generated_tokens = len(generated_ids)

        stopped_by_eos = (
            generated_tokens > 0
            and tokenizer.eos_token_id is not None
            and generated_ids[-1].item() == tokenizer.eos_token_id
        )

        results.append({
            "category": case["category"],
            "prompt": case["prompt"],
            "config_name": case["config_name"],
            "generation_config": case["generation_config"],
            "seed": case["seed"],
            "generated_text": tokenizer.decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            ),
            "prompt_tokens": prompt_tokens,
            "generated_tokens": generated_tokens,
            "stopped_by_eos": stopped_by_eos,
            "reached_token_limit": (
                generated_tokens == max_new_tokens
                and not stopped_by_eos
            ),
            "runtime_seconds": round(elapsed, 3),
            "tokens_per_second": round(
                generated_tokens / elapsed, 2
            ),
        })

    # Save results
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    filename = model_name.lower().replace("-", "_") + "_generation.json"
    comparison_dir = RESULTS_DIR / "comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)

    output_file = comparison_dir / filename

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            {
                "model": model_id,
                "max_new_tokens": baseline["max_new_tokens"],
                "results": results,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(f"\n{'=' * 60}")
    print(f"GENERATION COMPLETE — {model_name}")
    print(f"{'=' * 60}")
    print(f"Generations: {len(results)}")
    print(f"EOS stops:   {sum(r['stopped_by_eos'] for r in results)}")
    print(f"Output:      {output_file}")
    print(f"{'=' * 60}")

    return results

In [ ]:

MODEL_NAME = "GPT2-Mini"
MODEL_ID = MODEL_IDS[MODEL_NAME]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
).to(device).eval()

# GPT-2 has no dedicated padding token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print("MODEL COMPARISON — BASELINE LOADED")
print("=" * 60)
print(f"Model:      {MODEL_NAME}")
print(f"ID:         {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print(f"Context:    {model.config.max_position_embeddings:,}")
print("=" * 60)

comparison_results = run_generation_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)

In [ ]:

del model, tokenizer
gc.collect()

MODEL_NAME = "Pythia-70M"
MODEL_ID = MODEL_IDS[MODEL_NAME]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
).to(device).eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print("MODEL COMPARISON — BASELINE LOADED")
print("=" * 60)
print(f"Model:      {MODEL_NAME}")
print(f"ID:         {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print(f"Context:    {model.config.max_position_embeddings:,}")
print("=" * 60)

comparison_results = run_generation_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)

In [ ]:
del model, tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_NAME = "DistilGPT2"
MODEL_ID = MODEL_IDS[MODEL_NAME]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
).to(device).eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print("MODEL COMPARISON — BASELINE LOADED")
print("=" * 60)
print(f"Model:      {MODEL_NAME}")
print(f"ID:         {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print(f"Context:    {model.config.max_position_embeddings:,}")
print("=" * 60)

comparison_results = run_generation_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)

In [ ]:

MODEL_FILES = {
    "NeoLM": RESULTS_DIR / "neolm_generation.json",
    "GPT2-Mini": RESULTS_DIR / "comparison/gpt2_mini_generation.json",
    "Pythia-70M": RESULTS_DIR / "comparison/pythia_70m_generation.json",
    "DistilGPT2": RESULTS_DIR / "comparison/distilgpt2_generation.json",
}

frames = []

for name, path in MODEL_FILES.items():
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data["results"])
    df["model_name"] = name
    frames.append(df)

comparison_df = pd.concat(frames, ignore_index=True)

summary = comparison_df.groupby(
    "model_name", sort=False
).agg(
    generations=("generated_tokens", "size"),
    eos_stops=("stopped_by_eos", "sum"),
    avg_tokens=("generated_tokens", "mean"),
    avg_tokens_per_sec=("tokens_per_second", "mean"),
).round(2)

summary["eos_rate"] = (
    summary["eos_stops"] / summary["generations"] * 100
).round(2)

print("=" * 60)
print("SMALL LANGUAGE MODELS — GENERATION COMPARISON")
print("=" * 60)

display(summary)

In [ ]:
# Change these to explore other cases
selected_prompt = unique_prompts[0][1]
selected_config = "conservative"

samples = comparison_df[
    (comparison_df["prompt"] == selected_prompt)
    & (comparison_df["config_name"] == selected_config)
]

print("=" * 60)
print("GENERATION QUALITY COMPARISON")
print("=" * 60)
print("Prompt:", selected_prompt)
print("Config:", selected_config)

for _, row in samples.iterrows():
    print(f"\n{'=' * 60}")
    print(f"MODEL: {row['model_name']}")
    print(f"{'=' * 60}")
    print(row["generated_text"])
    print(
        f"\nTokens: {row['generated_tokens']} | "
        f"EOS: {row['stopped_by_eos']}"
    )

In [ ]:

HELLASWAG_SAMPLE_SIZE = 200

hellaswag = load_dataset(
    "Rowan/hellaswag",
    split="validation",
)

eval_sample = hellaswag.shuffle(seed=SEED).select(
    range(HELLASWAG_SAMPLE_SIZE)
)


def preprocess_hellaswag(text):
    text = text.strip().replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    return text.replace("  ", " ")


def prepare_hellaswag(row):
    context = row["ctx_a"] + " " + row["ctx_b"].capitalize()

    return {
        "context": preprocess_hellaswag(
            row["activity_label"] + ": " + context
        ),
        "endings": [
            preprocess_hellaswag(e)
            for e in row["endings"]
        ],
        "label": int(row["label"]),
    }


print("=" * 60)
print("HELLASWAG — COMPARISON SETUP")
print("=" * 60)
print(f"Examples:        {len(eval_sample):,}")
print(f"Choices/example: 4")
print(f"Random baseline: 25%")
print(f"Seed:            {SEED}")
print("=" * 60)

In [ ]:

@torch.inference_mode()
def score_ending(model, tokenizer, context, ending):
    context_ids = tokenizer.encode(
        context,
        add_special_tokens=False,
    )

    ending_ids = tokenizer.encode(
        " " + ending.strip(),
        add_special_tokens=False,
    )

    max_length = model.config.max_position_embeddings

    assert context_ids and ending_ids
    assert len(ending_ids) < max_length

    context_ids = context_ids[
        -(max_length - len(ending_ids)):
    ]

    input_ids = torch.tensor(
        [context_ids + ending_ids],
        dtype=torch.long,
        device=model.device,
    )

    logits = model(
        input_ids=input_ids,
        use_cache=False,
    ).logits[0]

    start = len(context_ids) - 1
    end = start + len(ending_ids)

    log_probs = F.log_softmax(
        logits[start:end].float(),
        dim=-1,
    )

    targets = torch.tensor(
        ending_ids,
        device=model.device,
    )

    token_scores = log_probs.gather(
        1, targets.unsqueeze(1)
    ).squeeze(1)

    return (
        token_scores.sum().item(),
        token_scores.mean().item(),
    )


def run_hellaswag_benchmark(model, tokenizer, model_name, model_id):
    raw_correct = 0
    norm_correct = 0

    start_time = time.perf_counter()

    print(f"\n{'=' * 60}")
    print(f"HELLASWAG BENCHMARK — {model_name}")
    print(f"{'=' * 60}")

    for row in tqdm(eval_sample, desc=model_name):
        example = prepare_hellaswag(row)

        scores = [
            score_ending(
                model,
                tokenizer,
                example["context"],
                ending,
            )
            for ending in example["endings"]
        ]

        raw_pred = int(np.argmax([s[0] for s in scores]))
        norm_pred = int(np.argmax([s[1] for s in scores]))

        raw_correct += raw_pred == example["label"]
        norm_correct += norm_pred == example["label"]

    elapsed = time.perf_counter() - start_time
    n = len(eval_sample)

    results = {
        "model": model_id,
        "benchmark": "hellaswag",
        "evaluation": "custom_zero_shot",
        "examples": n,
        "seed": SEED,
        "raw_accuracy": raw_correct / n,
        "normalized_accuracy": norm_correct / n,
        "raw_correct": raw_correct,
        "normalized_correct": norm_correct,
        "runtime_seconds": round(elapsed, 2),
    }

    output_dir = RESULTS_DIR / "comparison"
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = model_name.lower().replace("-", "_") + "_hellaswag.json"
    output_file = output_dir / filename

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"\n{'=' * 60}")
    print(f"HELLASWAG RESULTS — {model_name}")
    print(f"{'=' * 60}")
    print(f"Examples:            {n:,}")
    print(f"Raw accuracy:        {results['raw_accuracy']:.2%}")
    print(f"Normalized accuracy: {results['normalized_accuracy']:.2%}")
    print(f"Raw correct:         {raw_correct}/{n}")
    print(f"Normalized correct:  {norm_correct}/{n}")
    print(f"Runtime:             {elapsed:.1f}s")
    print(f"Saved to:            {output_file}")
    print(f"{'=' * 60}")

    return results

In [ ]:
hellaswag_results = run_hellaswag_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)

In [ ]:


del model, tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_NAME = "Pythia-70M"
MODEL_ID = MODEL_IDS[MODEL_NAME]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print(f"MODEL:      {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {model.device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print("=" * 60)

hellaswag_results = run_hellaswag_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)

In [ ]:
del model, tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_NAME = "GPT2-Mini"
MODEL_ID = MODEL_IDS[MODEL_NAME]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
).to(device).eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print("MODEL COMPARISON — BASELINE LOADED")
print("=" * 60)
print(f"Model:      {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {model.device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print("=" * 60)

hellaswag_results = run_hellaswag_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)

In [ ]:
del model, tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_NAME = "NeoLM"
MODEL_ID = "MaagDeveloper/NeoLM-Small-Base"

tokenizer = AutoTokenizer.from_pretrained(
    "../artifacts/model",
    trust_remote_code=True,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "../artifacts/model",
    trust_remote_code=True,
    local_files_only=True,
).to(device).eval()

print("=" * 60)
print("MODEL COMPARISON — NEOLM")
print("=" * 60)
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {model.device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print("=" * 60)

hellaswag_results = run_hellaswag_benchmark(
    model=model,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    model_id=MODEL_ID,
)


## Research Conclusion

We compared NeoLM-Small-Base (61.4M) against three similarly sized pretrained base language models.

### Key Findings

- **NeoLM** achieved the highest raw HellaSwag accuracy (31.5%) on the 200-example sample.
- **DistilGPT2** achieved the highest length-normalized accuracy (31.0%).
- NeoLM demonstrated readable story continuations but struggled with factual accuracy, reasoning, and code generation.
- GPT2-Mini showed substantial repetition in some generations.
- No model demonstrated consistently strong performance across all evaluated tasks.

### Limitations

- HellaSwag used a custom scoring implementation and only 200 examples.
- Generation quality was assessed qualitatively.
- Training budgets, tokenizers, GPU environments, and model precision differ.
- Small accuracy differences do not establish a statistically reliable winner.

### Conclusion

NeoLM successfully demonstrates an end-to-end language model development pipeline, from corpus preparation and tokenizer training to pretraining, inference, and evaluation.

The released 61M model exhibits meaningful language modeling capabilities while remaining limited in reasoning and factual reliability.

These findings establish a baseline for future NeoLM experiments, not the performance ceiling of the architecture.
